# LA Studio Voice GPU Worker

Select **Runtime > Change runtime type > GPU** and then choose **Run all**. This notebook runs a direct temporary Colab worker for normal Kokoro TTS. It never reads, stores, or calls an API Gateway key. The final cell prints a temporary HTTPS URL and bearer token to enter in LA Studio's **Colab GPU TTS** panel.

The Kova Voice Studio notebook is a separate profile-based OmniVoice worker for voice cloning; use that only for the Phase 7 clone workflow.

In [ ]:
import subprocess, sys

def run(*args):
    print('+', ' '.join(args))
    subprocess.run(args, check=True)

run('nvidia-smi')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'kokoro>=0.9.4', 'soundfile>=0.12.1', 'fastapi>=0.115.0', 'uvicorn[standard]>=0.34.0')


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_worker.py')
WORKER.write_text(r'''
import io
import os
import threading
from functools import lru_cache

import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field
from kokoro import KPipeline

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before starting this worker.')

TOKEN = os.environ['LA_STUDIO_COLAB_TOKEN']
# Only advertise models/languages that this notebook actually loads. Add a
# separate tested adapter before exposing other LA Studio TTS families.
LANGUAGE_CODES = {'en': 'a', 'en-us': 'a', 'en-gb': 'b', 'ja': 'j', 'zh': 'z', 'es': 'e', 'fr': 'f', 'hi': 'h', 'it': 'i', 'pt-br': 'p'}
SUPPORTED_VOICES = ['af_heart', 'af_bella', 'af_nicole', 'am_adam', 'am_michael', 'bf_emma', 'bm_george']
MAX_INPUT_CHARS = 4000
MAX_OUTPUT_SECONDS = 300
SYNTHESIS_SLOTS = threading.BoundedSemaphore(1)

class SpeechRequest(BaseModel):
    model: str = Field(min_length=1, max_length=120)
    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)
    voice: str = Field(min_length=1, max_length=120)
    language: str = 'en'
    speed: float = Field(default=1.0, ge=0.25, le=4.0)
    response_format: str = 'wav'
    settings: dict = Field(default_factory=dict)

def require_token(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

@lru_cache(maxsize=8)
def pipeline_for(language_code: str):
    pipeline = KPipeline(lang_code=language_code)
    # Kokoro releases expose the torch model on different attributes. Move
    # whichever one is present to CUDA without relying on a private API.
    for name in ('model', 'kokoro_model'):
        model = getattr(pipeline, name, None)
        if hasattr(model, 'to'):
            model.to('cuda')
    return pipeline

app = FastAPI(title='LA Studio Colab TTS Worker', docs_url=None, redoc_url=None, openapi_url=None)

@app.get('/health')
@app.get('/v1/health')
def health(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'status': 'ready', 'ready': True, 'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'api_version': '1.0'}

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {'contract_version': 1, 'capabilities': [{'id': 'tts', 'models': [{'id': 'kokoro', 'languages': sorted(LANGUAGE_CODES), 'voices': SUPPORTED_VOICES, 'formats': ['wav'], 'device': 'cuda'}]}]}

@app.post('/v1/audio/speech')
def speech(request: SpeechRequest, authorization: str | None = Header(default=None)):
    require_token(authorization)
    if request.model.lower() != 'kokoro':
        raise HTTPException(status_code=422, detail='this worker currently supports model kokoro only')
    language = request.language.strip().lower() or 'en'
    language_code = LANGUAGE_CODES.get(language)
    if not language_code:
        raise HTTPException(status_code=422, detail='language is not supported by this Kokoro worker')
    if request.voice not in SUPPORTED_VOICES:
        raise HTTPException(status_code=422, detail='voice is not supported by this Kokoro worker')
    if request.response_format.lower() != 'wav':
        raise HTTPException(status_code=422, detail='only WAV output is supported')
    if not SYNTHESIS_SLOTS.acquire(blocking=False):
        raise HTTPException(status_code=429, detail='the Colab TTS worker is busy; retry shortly')
    try:
        chunks = []
        for _, _, audio in pipeline_for(language_code)(request.input, voice=request.voice, speed=request.speed):
            samples = np.asarray(audio, dtype=np.float32).reshape(-1)
            if samples.size:
                chunks.append(samples)
        if not chunks:
            raise RuntimeError('Kokoro returned no audio')
        output = np.concatenate(chunks)
        if output.size > 24000 * MAX_OUTPUT_SECONDS:
            raise HTTPException(status_code=413, detail='generated audio exceeds the five minute output limit')
        if not np.isfinite(output).all() or np.max(np.abs(output)) > 1.2:
            raise RuntimeError('Kokoro returned invalid audio samples')
        buffer = io.BytesIO()
        sf.write(buffer, output, 24000, format='WAV', subtype='PCM_16')
        return Response(buffer.getvalue(), media_type='audio/wav', headers={'Cache-Control': 'no-store'})
    except HTTPException:
        raise
    except Exception as error:
        raise HTTPException(status_code=503, detail='Colab Kokoro synthesis failed: ' + type(error).__name__) from error
    finally:
        SYNTHESIS_SLOTS.release()
''', encoding='utf-8')
print(WORKER)


In [ ]:
import os, re, secrets, subprocess, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_TOKEN'] = TOKEN
worker = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'la_studio_voice_worker:app', '--host', '127.0.0.1', '--port', '3921'], cwd='/content', env=env)
for _ in range(30):
    try:
        request = urllib.request.Request('http://127.0.0.1:3921/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('LA Studio TTS worker did not become ready')
subprocess.run(['bash', '-lc', 'wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb'], check=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3921', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate(); tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')
print('\nLA_STUDIO_COLAB_TTS_URL=' + public_url)
print('LA_STUDIO_COLAB_TTS_TOKEN=' + TOKEN)
print('MODEL=kokoro  VOICE=af_heart  LANGUAGE=en')
